# Checking if the data changes through the URL and updates itself every 60 s

Using the `gtfs_realtime_pb2` library to read the `.pb` file (GTFS-RT, protobuf format).

In [33]:
from google.transit import gtfs_realtime_pb2  # compiled GTFS-RT schema (from gtfs-realtime-bindings)
import urllib.request                            # download the feed
import pandas as pd                              # tables


Requesting the `realtime-free.pb` file from https://realtime.gtfs.de/realtime-free.pb

In [34]:
# Realtime feed from gtfs.de: nationwide, rebuilt once per minute
URL = "https://realtime.gtfs.de/realtime-free.pb"
urllib.request.urlretrieve(URL, "data/realtime-free.pb")  # local copy for testing

('data/realtime-free.pb', <http.client.HTTPMessage at 0x16e465010>)

Exploring the data:


In [35]:
try:
    req = urllib.request.Request(URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as response:
        feed = gtfs_realtime_pb2.FeedMessage()
        feed.ParseFromString(response.read())  # bytes -> Python objects (deserialize)
        rows = []

    # entity = one trip; build one row per stop of each trip
    for entity in feed.entity:
        if entity.HasField("trip_update"):
            for stop in entity.trip_update.stop_time_update:
                rows.append({
                    "trip_id": entity.trip_update.trip.trip_id,
                    "stop_id": stop.stop_id,
                    # delays in seconds (positive = late, negative = early); None = no data
                    "arrival_delay": stop.arrival.delay if stop.HasField("arrival") else None,
                    "departure_delay": stop.departure.delay if stop.HasField("departure") else None,
                    "timestamp": feed.header.timestamp  # when gtfs.de built the feed (same for all rows)
                })

except Exception as e:
    print(f"Error: {e}")
df = pd.DataFrame(rows)  # realtime table
feed_time = pd.to_datetime(feed.header.timestamp, unit="s", utc=True).tz_convert("Europe/Berlin")
# Unix seconds -> Berlin time (HH:MM:SS)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True).dt.tz_convert("Europe/Berlin").dt.strftime("%H:%M:%S")
print("Rows:", len(df))
df.head()


Rows: 302461


,trip_id,stop_id,arrival_delay,departure_delay,timestamp
0,633479,646989,NaN,20.0,19:14:49
1,633479,39669,1.0,1.0,19:14:49
2,633479,55844,18.0,18.0,19:14:49
3,633479,351315,-60.0,-60.0,19:14:49
4,633479,103945,-16.0,-16.0,19:14:49


In [36]:
# agency_id 474 = BOGESTRA (Bochum-Gelsenkirchener Straßenbahnen AG)
routes = pd.read_csv("data/routes.txt")
bogestra_routes = routes[(routes["agency_id"] == 474 )]["route_id"]  # all Bogestra route_ids
bogestra_routes.info()
bogestra_routes.head()

<class 'pandas.Series'>
Index: 190 entries, 663 to 24454
Series name: route_id
Non-Null Count  Dtype
--------------  -----
190 non-null    int64
dtypes: int64(1)
memory usage: 3.0 KB


663     11675
3088    15118
6143     6623
6171     7944
6178    22916
Name: route_id, dtype: int64

In [37]:
# All scheduled Bogestra trips (static timetable, all days)
trips = pd.read_csv("data/trips.txt")
bogestra_trips = trips[(trips['route_id'].isin(bogestra_routes))]["trip_id"]
bogestra_trips.info()

<class 'pandas.Series'>
Index: 25032 entries, 17377 to 1829021
Series name: trip_id
Non-Null Count  Dtype
--------------  -----
25032 non-null  int64
dtypes: int64(1)
memory usage: 391.1 KB


In [ ]:
# Bogestra rows in the realtime feed (trip_id is text there -> astype(str))
bogestra_df = df[df['trip_id'].isin(bogestra_trips.astype(str))]
print("Rows:", len(bogestra_df))                          # stop events (trip x stop)
print("Unique Trips:", bogestra_df["trip_id"].nunique())  # running trips
bogestra_df.head()

Rows: 4163
Unique Trips: 659


,trip_id,stop_id,arrival_delay,departure_delay,timestamp
717,889713,582337,NaN,197.0,19:14:49
718,889713,310319,257.0,258.0,19:14:49
719,889713,353012,159.0,160.0,19:14:49
720,889713,39987,97.0,NaN,19:14:49
1115,435913,675173,NaN,0.0,19:14:49
